In [24]:
# ==========================================
# CELL 1: ENVIRONMENT SETUP
# ==========================================
!pip install transformers datasets timm accelerate -q

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as transforms
from PIL import Image
import json
import os
import re
import random
import numpy as np
from collections import Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive (for saving models)
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔥 Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔥 Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB


In [25]:
# ==========================================
# CELL 2: CONFIGURATION (UPDATED WITH DRIVE CHECKPOINTS)
# ==========================================
import os

class Config:
    # Data paths
    DATA_DIR = "/content/drive/MyDrive/Deep Learning Project/2 words Q&As/Raw Q&As"
    IMAGE_DIR = "/content/drive/MyDrive/Deep Learning Project/unzipped_images/raw"

    # CHECKPOINTS NOW SAVE TO DRIVE (not local!)
    CHECKPOINT_DIR = "/content/drive/MyDrive/vqa_checkpoints"

    # Model architecture
    VIT_MODEL = 'google/vit-base-patch16-224'
    BERT_MODEL = 'bert-base-uncased'

    # Image settings
    IMG_SIZE = 224
    PATCH_SIZE = 16

    # Text settings
    MAX_QUESTION_LEN = 64

    # Fusion settings
    FUSION_SIZE = 512

    # Training
    BATCH_SIZE = 32
    LEARNING_RATE = 3e-5
    WEIGHT_DECAY = 0.01
    NUM_EPOCHS = 30
    WARMUP_EPOCHS = 2
    EARLY_STOPPING = 7

    # Regularization
    DROPOUT = 0.3
    LABEL_SMOOTHING = 0.1

    # Data
    MIN_ANSWER_FREQ = 5
    NUM_WORKERS = 2

    # Mixed precision
    USE_AMP = True

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) #gradient clipping


    def __init__(self):
        # Create checkpoint directory on Drive
        os.makedirs(self.CHECKPOINT_DIR, exist_ok=True)
        print(f"✅ Checkpoint directory: {self.CHECKPOINT_DIR}")

        # Verify data paths exist
        if os.path.exists(self.DATA_DIR):
            print(f"✅ Data directory: {self.DATA_DIR}")
        else:
            print(f"⚠️ Data directory not found: {self.DATA_DIR}")

        if os.path.exists(self.IMAGE_DIR):
            print(f"✅ Image directory: {self.IMAGE_DIR}")
        else:
            print(f"⚠️ Image directory not found: {self.IMAGE_DIR}")

# Initialize config
cfg = Config()
print(f"✅ Config loaded | Batch: {cfg.BATCH_SIZE} | LR: {cfg.LEARNING_RATE}")

✅ Checkpoint directory: /content/drive/MyDrive/vqa_checkpoints
✅ Data directory: /content/drive/MyDrive/Deep Learning Project/2 words Q&As/Raw Q&As
✅ Image directory: /content/drive/MyDrive/Deep Learning Project/unzipped_images/raw
✅ Config loaded | Batch: 32 | LR: 3e-05


In [26]:
# ==========================================
# CELL 3: DATA PREPROCESSING & VOCABULARY
# ==========================================
import glob  # Add this import

def clean_answer(text):
    text = str(text).strip().lower()
    text = re.sub(r'\b(one|1)\b', '1', text)
    text = re.sub(r'\b(two|2)\b', '2', text)
    text = re.sub(r'\b(three|3)\b', '3', text)
    text = re.sub(r'\b(four|4)\b', '4', text)
    text = re.sub(r'\b(five|5)\b', '5', text)
    text = re.sub(r'\s+', ' ', text)
    return text

def load_data(data_dir):
    """Load JSONL files from directory (including subfolders)"""
    all_samples = []

    if not os.path.exists(data_dir):
        print(f"⚠️ Directory not found: {data_dir}")
        print("Please upload your JSONL files to this location")
        return []

    # Recursively find all JSONL files
    jsonl_files = glob.glob(os.path.join(data_dir, "**/*.jsonl"), recursive=True)
    jsonl_files.extend(glob.glob(os.path.join(data_dir, "**/*.json"), recursive=True))

    if not jsonl_files:
        jsonl_files = [f for f in os.listdir(data_dir) if f.endswith(('.jsonl', '.json'))]
        jsonl_files = [os.path.join(data_dir, f) for f in jsonl_files]

    print(f"Found {len(jsonl_files)} JSON/JSONL files")

    if len(jsonl_files) == 0:
        print(f"   No JSON files found in {data_dir}")
        return []

    for file_path in tqdm(jsonl_files, desc="Loading data"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                for line_num, line in enumerate(f):
                    if not line.strip():
                        continue
                    try:
                        item = json.loads(line.strip())

                        # Handle different JSON structures
                        image_id = item.get('image_id') or item.get('id') or f"img_{line_num}"

                        # Get QA pairs (handle different field names)
                        qa_pairs = item.get('qa_pairs') or item.get('qas') or []

                        for qa in qa_pairs:
                            question = qa.get('question') or qa.get('q') or ''
                            answer = qa.get('answer') or qa.get('a') or ''

                            if question and answer:
                                all_samples.append({
                                    'image_id': str(image_id),
                                    'question': question.strip(),
                                    'answer': clean_answer(answer)
                                })
                    except json.JSONDecodeError:
                        continue
        except Exception as e:
            print(f"   Error reading {os.path.basename(file_path)}: {e}")

    print(f"✅ Loaded {len(all_samples)} Q&A pairs")

    # Show sample
    if len(all_samples) > 0:
        print(f"\n📋 Sample:")
        print(f"   Image ID: {all_samples[0]['image_id']}")
        print(f"   Question: {all_samples[0]['question'][:80]}...")
        print(f"   Answer: {all_samples[0]['answer']}")

    return all_samples

# Build vocabulary
all_samples = load_data(cfg.DATA_DIR)

if len(all_samples) > 0:
    answer_counts = Counter([s['answer'] for s in all_samples])
    frequent_answers = [ans for ans, count in answer_counts.items()
                       if count >= cfg.MIN_ANSWER_FREQ]

    answer_to_idx = {ans: i for i, ans in enumerate(frequent_answers)}
    idx_to_answer = {i: ans for ans, i in answer_to_idx.items()}

    # Add UNK token
    UNK_TOKEN = "<unk>"
    answer_to_idx[UNK_TOKEN] = len(answer_to_idx)
    idx_to_answer[len(answer_to_idx)-1] = UNK_TOKEN

    NUM_ANSWERS = len(answer_to_idx)

    # Convert samples to indices
    unk_count = 0
    for s in all_samples:
        if s['answer'] in answer_to_idx:
            s['answer_idx'] = answer_to_idx[s['answer']]
        else:
            s['answer_idx'] = answer_to_idx[UNK_TOKEN]
            unk_count += 1

    print(f"\n📊 Vocabulary size: {NUM_ANSWERS} (including <unk>)")
    print(f"   UNK rate: {unk_count}/{len(all_samples)} ({unk_count/len(all_samples):.1%})")
    print(f"   Top 10 answers: {list(frequent_answers[:10])}")
else:
    NUM_ANSWERS = 100  # Placeholder
    print("⚠️ No data loaded! Please check your data path.")

Found 14 JSON/JSONL files


Loading data: 100%|██████████| 14/14 [00:00<00:00, 14.83it/s]


✅ Loaded 24656 Q&A pairs

📋 Sample:
   Image ID: pexels_6248997
   Question: Is there a pot on the stove?...
   Answer: yes

📊 Vocabulary size: 265 (including <unk>)
   UNK rate: 5745/24656 (23.3%)
   Top 10 answers: ['yes', 'multiple', 'red', 'not specified', 'no', '1', 'white', '2', 'on the stove', 'preparing food']


In [27]:
# ==========================================
# CELL 4: DATASET CLASS
# ==========================================
from transformers import BertTokenizer

class VQADataset(Dataset):
    def __init__(self, samples, image_dir, split='train'):
        self.samples = samples
        self.image_dir = image_dir
        self.split = split
        self.tokenizer = BertTokenizer.from_pretrained(cfg.BERT_MODEL)

        # Image transforms
        if split == 'train':
            self.transform = transforms.Compose([
                transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225])
            ])

        # Cache image paths
        self.image_cache = {}
        if os.path.exists(image_dir):
            for f in os.listdir(image_dir):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_id = os.path.splitext(f)[0]
                    self.image_cache[img_id] = os.path.join(image_dir, f)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Load image
        image_path = self.image_cache.get(sample['image_id'])
        if image_path and os.path.exists(image_path):
            image = Image.open(image_path).convert('RGB')
        else:
            image = Image.new('RGB', (cfg.IMG_SIZE, cfg.IMG_SIZE))

        image = self.transform(image)

        # Tokenize question
        tokens = self.tokenizer(
            sample['question'],
            max_length=cfg.MAX_QUESTION_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'image': image,
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'answer': torch.tensor(sample['answer_idx'], dtype=torch.long)
        }


In [28]:
# ==========================================
# CELL 5: VISION TRANSFORMER ENCODER
# ==========================================
from transformers import ViTModel

class ViTEncoder(nn.Module):
    def __init__(self, model_name=cfg.VIT_MODEL):
        super().__init__()
        self.vit = ViTModel.from_pretrained(model_name)
        hidden_size = self.vit.config.hidden_size

        self.projection = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, cfg.FUSION_SIZE),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT)
        )

    def forward(self, pixel_values):
        # ViT outputs: last_hidden_state (batch, seq_len, hidden)
        outputs = self.vit(pixel_values)
        # Use CLS token
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return self.projection(cls_embedding)

In [29]:
# ==========================================
# CELL 6: BERT TEXT ENCODER
# ==========================================
from transformers import BertModel

class BertTextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained(cfg.BERT_MODEL)
        hidden_size = self.bert.config.hidden_size

        self.projection = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, cfg.FUSION_SIZE),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Use CLS token
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return self.projection(cls_embedding)


In [30]:
# ==========================================
# CELL 7: MULTIMODAL FUSION
# ==========================================
class MultimodalFusion(nn.Module):
    def __init__(self):
        super().__init__()

        # Cross-attention (bidirectional)
        self.cross_attn_v2t = nn.MultiheadAttention(
            cfg.FUSION_SIZE, num_heads=8, dropout=cfg.DROPOUT, batch_first=True
        )
        self.cross_attn_t2v = nn.MultiheadAttention(
            cfg.FUSION_SIZE, num_heads=8, dropout=cfg.DROPOUT, batch_first=True
        )

        # Self-attention for each modality
        self.self_attn_v = nn.MultiheadAttention(
            cfg.FUSION_SIZE, num_heads=8, dropout=cfg.DROPOUT, batch_first=True
        )
        self.self_attn_t = nn.MultiheadAttention(
            cfg.FUSION_SIZE, num_heads=8, dropout=cfg.DROPOUT, batch_first=True
        )

        # Fusion layers
        self.fusion_mlp = nn.Sequential(
            nn.Linear(cfg.FUSION_SIZE * 2, cfg.FUSION_SIZE),
            nn.LayerNorm(cfg.FUSION_SIZE),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(cfg.FUSION_SIZE, cfg.FUSION_SIZE // 2),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT)
        )

    def forward(self, v, t):
        # Add sequence dimension for attention (batch, 1, dim)
        v_seq = v.unsqueeze(1)
        t_seq = t.unsqueeze(1)

        # Self-attention
        v_self, _ = self.self_attn_v(v_seq, v_seq, v_seq)
        t_self, _ = self.self_attn_t(t_seq, t_seq, t_seq)

        # Cross-attention
        v_cross, _ = self.cross_attn_v2t(v_self, t_self, t_self)
        t_cross, _ = self.cross_attn_t2v(t_self, v_self, v_self)

        # Combine and squeeze
        v_out = (v_self + v_cross).squeeze(1)
        t_out = (t_self + t_cross).squeeze(1)

        # Concatenate and fuse
        fused = torch.cat([v_out, t_out], dim=1)
        return self.fusion_mlp(fused)


In [31]:
# ==========================================
# CELL 8: COMPLETE VQA MODEL
# ==========================================
class ViTBertVQA(nn.Module):
    def __init__(self, num_answers):
        super().__init__()
        self.vision_encoder = ViTEncoder()
        self.text_encoder = BertTextEncoder()
        self.fusion = MultimodalFusion()
        self.classifier = nn.Linear(cfg.FUSION_SIZE // 2, num_answers)

    def forward(self, image, input_ids, attention_mask):
        v = self.vision_encoder(image)
        t = self.text_encoder(input_ids, attention_mask)
        fused = self.fusion(v, t)
        return self.classifier(fused)


In [32]:
# ==========================================
# CELL 9: TRAINING SETUP
# ==========================================
def create_dataloaders(all_samples, image_dir):
    """Split data and create dataloaders"""
    # Image-level split
    unique_images = list(set(s['image_id'] for s in all_samples))
    random.seed(42)
    random.shuffle(unique_images)

    n = len(unique_images)
    train_imgs = set(unique_images[:int(n * 0.7)])
    val_imgs = set(unique_images[int(n * 0.7):int(n * 0.85)])
    test_imgs = set(unique_images[int(n * 0.85):])

    train_samples = [s for s in all_samples if s['image_id'] in train_imgs]
    val_samples = [s for s in all_samples if s['image_id'] in val_imgs]
    test_samples = [s for s in all_samples if s['image_id'] in test_imgs]

    print(f"📊 Split: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}")

    train_dataset = VQADataset(train_samples, image_dir, 'train')
    val_dataset = VQADataset(val_samples, image_dir, 'val')
    test_dataset = VQADataset(test_samples, image_dir, 'test')

    train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE,
                              shuffle=True, num_workers=cfg.NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE,
                            shuffle=False, num_workers=cfg.NUM_WORKERS)
    test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE,
                             shuffle=False, num_workers=cfg.NUM_WORKERS)

    return train_loader, val_loader, test_loader

# Initialize model
if len(all_samples) > 0:
    model = ViTBertVQA(NUM_ANSWERS).to(device)
    print(f"🏗️ Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Loss with label smoothing
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)

    # Optimizer with layer-wise decay
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=cfg.LEARNING_RATE,
                                  weight_decay=cfg.WEIGHT_DECAY)

    # Cosine scheduler with warmup
    from transformers import get_cosine_schedule_with_warmup

    train_loader, val_loader, test_loader = create_dataloaders(all_samples, cfg.IMAGE_DIR)

    total_steps = len(train_loader) * cfg.NUM_EPOCHS
    warmup_steps = len(train_loader) * cfg.WARMUP_EPOCHS

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    scaler = GradScaler(enabled=cfg.USE_AMP)

    print("✅ Training setup complete!")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🏗️ Model parameters: 201,589,769
📊 Split: Train=17272, Val=3680, Test=3704
✅ Training setup complete!


In [34]:
# ==========================================
# CELL 10: TRAINING LOOP
# ==========================================
def train_epoch(model, loader, optimizer, criterion, scheduler, scaler):
    model.train()
    total_loss = 0
    predictions, targets = [], []

    pbar = tqdm(loader, desc="Training")
    for batch in pbar:
        images = batch['image'].to(device)
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        answers = batch['answer'].to(device)

        with autocast(enabled=cfg.USE_AMP):
            logits = model(images, input_ids, mask)
            loss = criterion(logits, answers)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        predictions.extend(preds.cpu().numpy())
        targets.extend(answers.cpu().numpy())

        pbar.set_postfix({'loss': f'{loss.item():.3f}'})

    acc = accuracy_score(targets, predictions)
    return total_loss / len(loader), acc

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    predictions, targets = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            images = batch['image'].to(device)
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            answers = batch['answer'].to(device)

            with autocast(enabled=cfg.USE_AMP):
                logits = model(images, input_ids, mask)
                loss = criterion(logits, answers)

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            predictions.extend(preds.cpu().numpy())
            targets.extend(answers.cpu().numpy())

    acc = accuracy_score(targets, predictions)
    return total_loss / len(loader), acc

# Training loop
if len(all_samples) > 0:
    print("\n🚀 Starting Training...\n")

    best_val_acc = 0
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(cfg.NUM_EPOCHS):
        print(f"\n📚 Epoch {epoch+1}/{cfg.NUM_EPOCHS}")

        train_loss, train_acc = train_epoch(model, train_loader, optimizer,
                                            criterion, scheduler, scaler)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"📊 Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"📊 Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'answer_to_idx': answer_to_idx,
                'idx_to_answer': idx_to_answer,
                'config': cfg,
            }, os.path.join(cfg.CHECKPOINT_DIR, 'best_model.pt'))
            print(f"💾 Saved best model (Acc: {val_acc:.4f})")
        else:
            patience_counter += 1

        if patience_counter >= cfg.EARLY_STOPPING:
            print(f"⏹️ Early stopping at epoch {epoch+1}")
            break

    print(f"\n🏆 Best Validation Accuracy: {best_val_acc:.4f}")

    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'], label='Val Loss')
    axes[0].legend()
    axes[0].set_title('Loss')
    axes[0].grid(True)

    axes[1].plot(history['train_acc'], label='Train Acc')
    axes[1].plot(history['val_acc'], label='Val Acc')
    axes[1].legend()
    axes[1].set_title('Accuracy')
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(cfg.CHECKPOINT_DIR, 'training_curves.png'))
    plt.show()



🚀 Starting Training...


📚 Epoch 1/30


Evaluating: 100%|██████████| 115/115 [02:46<00:00,  1.44s/it]


📊 Train Loss: 4.5764 | Train Acc: 0.2550
📊 Val Loss: 3.1259 | Val Acc: 0.5098
💾 Saved best model (Acc: 0.5098)

📚 Epoch 2/30


Evaluating: 100%|██████████| 115/115 [00:56<00:00,  2.05it/s]


📊 Train Loss: 2.8911 | Train Acc: 0.5358
📊 Val Loss: 2.5467 | Val Acc: 0.5639
💾 Saved best model (Acc: 0.5639)

📚 Epoch 3/30


...

📊 Train Loss: 1.6695 | Train Acc: 0.7944
📊 Val Loss: 2.2960 | Val Acc: 0.6353

📚 Epoch 16/30


Evaluating: 100%|██████████| 115/115 [00:57<00:00,  2.00it/s]


📊 Train Loss: 1.6340 | Train Acc: 0.8045
📊 Val Loss: 2.2951 | Val Acc: 0.6418

📚 Epoch 17/30


Training:  84%|████████▎ | 452/540 [05:45<01:07,  1.31it/s, loss=1.404]


KeyboardInterrupt: 

In [38]:
# ==========================================
# CELL 11: TEST EVALUATION (FIXED)
# ==========================================
if len(all_samples) > 0:
    # Load best model with weights_only=False
    checkpoint_path = os.path.join(cfg.CHECKPOINT_DIR, 'best_model.pt')

    if os.path.exists(checkpoint_path):
        # Fixed: added weights_only=False
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()

        print(f"\n🧪 Loading best model from epoch {checkpoint['epoch']+1}")
        print(f"   Validation accuracy during training: {checkpoint['val_acc']:.2%}")

        # Test evaluation
        test_loss, test_acc = evaluate(model, test_loader, criterion)

        print("\n" + "="*60)
        print(f"🎯 FINAL TEST RESULTS")
        print("="*60)
        print(f"   Test Accuracy: {test_acc:.4f} ({test_acc:.2%})")
        print(f"   Test Loss: {test_loss:.4f}")

        # Top-K accuracy
        try:
            from torchmetrics.classification import Accuracy

            top1 = Accuracy(task="multiclass", num_classes=NUM_ANSWERS, top_k=1).to(device)
            top3 = Accuracy(task="multiclass", num_classes=NUM_ANSWERS, top_k=3).to(device)
            top5 = Accuracy(task="multiclass", num_classes=NUM_ANSWERS, top_k=5).to(device)

            model.eval()
            with torch.no_grad():
                for batch in tqdm(test_loader, desc="Computing Top-K"):
                    images = batch['image'].to(device)
                    input_ids = batch['input_ids'].to(device)
                    mask = batch['attention_mask'].to(device)
                    answers = batch['answer'].to(device)

                    logits = model(images, input_ids, mask)
                    top1.update(logits, answers)
                    top3.update(logits, answers)
                    top5.update(logits, answers)

            print(f"\n📊 Top-K Accuracy:")
            print(f"   Top-1: {top1.compute():.4f} ({top1.compute():.2%})")
            print(f"   Top-3: {top3.compute():.4f} ({top3.compute():.2%})")
            print(f"   Top-5: {top5.compute():.4f} ({top5.compute():.2%})")
        except ImportError:
            print("   Install torchmetrics for Top-K: !pip install torchmetrics")

        # Save results
        with open(os.path.join(cfg.CHECKPOINT_DIR, 'test_results.txt'), 'w') as f:
            f.write(f"Test Accuracy: {test_acc:.4f}\n")
            f.write(f"Test Loss: {test_loss:.4f}\n")
            f.write(f"Best Validation Accuracy: {checkpoint['val_acc']:.4f}\n")

        print(f"\n💾 Results saved to {cfg.CHECKPOINT_DIR}/test_results.txt")
    else:
        print(f"\n⚠️ No checkpoint found at {checkpoint_path}")
        print("   Train the model first!")


🧪 Loading best model from epoch 11
   Validation accuracy during training: 64.40%


Evaluating: 100%|██████████| 116/116 [02:48<00:00,  1.46s/it]


🎯 FINAL TEST RESULTS
   Test Accuracy: 0.6490 (64.90%)
   Test Loss: 2.2286
   Install torchmetrics for Top-K: !pip install torchmetrics

💾 Results saved to /content/drive/MyDrive/vqa_checkpoints/test_results.txt


In [43]:
# ==========================================
# CELL 12: INFERENCE DEMO (FIXED)
# ==========================================
from IPython.display import display, clear_output, Image as IPImage
import ipywidgets as widgets

def predict_single(image_path, question, top_k=5):
    """Predict answer for a single image-question pair"""
    model.eval()

    # Load and transform image
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0).to(device)

    # Tokenize question
    tokens = tokenizer(question, max_length=cfg.MAX_QUESTION_LEN,
                      padding='max_length', truncation=True, return_tensors='pt')
    input_ids = tokens['input_ids'].to(device)
    attention_mask = tokens['attention_mask'].to(device)

    with torch.no_grad():
        with autocast(enabled=cfg.USE_AMP):
            logits = model(image, input_ids, attention_mask)
            probs = F.softmax(logits, dim=1)

    top_probs, top_indices = torch.topk(probs, top_k, dim=1)

    results = []
    for i in range(top_k):
        idx = top_indices[0][i].item()
        results.append({
            'answer': idx_to_answer.get(idx, '<unk>'),
            'confidence': top_probs[0][i].item()
        })

    return results

if len(all_samples) > 0:
    # Load best model with weights_only=False
    checkpoint_path = os.path.join(cfg.CHECKPOINT_DIR, 'best_model.pt')

    if os.path.exists(checkpoint_path):
        # Fixed: added weights_only=False
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()

        # Use vocabulary from checkpoint
        idx_to_answer = checkpoint['idx_to_answer']

        tokenizer = BertTokenizer.from_pretrained(cfg.BERT_MODEL)
        transform = transforms.Compose([
            transforms.Resize((cfg.IMG_SIZE, cfg.IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])

        print("\n" + "="*60)
        print(f"🎯 VQA Demo Ready!")
        print(f"   Best model from epoch {checkpoint['epoch']+1}")
        print(f"   Validation accuracy: {checkpoint['val_acc']:.2%}")
        print("="*60)

        # Create widgets
        uploader = widgets.FileUpload(accept='image/*', multiple=False,
                                      description='📁 Upload Image')
        question_input = widgets.Text(placeholder="Ask a question about the image...",
                                      description="❓ Question:")
        predict_button = widgets.Button(description="🔍 Predict", button_style='success')
        output = widgets.Output()

        def on_predict(b):
            with output:
                clear_output()
                if not uploader.value:
                    print("⚠️ Please upload an image first")
                    return
                if not question_input.value:
                    print("⚠️ Please enter a question")
                    return

                # Save uploaded image
                uploaded = list(uploader.value.values())[0]
                img_path = "/tmp/temp_image.jpg"
                with open(img_path, 'wb') as f:
                    f.write(uploaded['content'])

                # Display image
                display(IPImage(img_path, width=300))

                # Predict
                results = predict_single(img_path, question_input.value)

                print(f"\n❓ Question: {question_input.value}")
                print("\n🏆 Top 5 Predictions:")
                medals = ['🥇', '🥈', '🥉', '4️⃣', '5️⃣']
                for i, r in enumerate(results):
                    bar = '█' * int(r['confidence'] * 30)
                    print(f"   {medals[i]} {r['answer']:<25} {r['confidence']:.1%} {bar}")

        predict_button.on_click(on_predict)

        display(widgets.VBox([
            widgets.HTML("<h3>📷 Upload an image and ask a question</h3>"),
            uploader,
            question_input,
            predict_button,
            output
        ]))
    else:
        print(f"⚠️ No trained model found at {checkpoint_path}")
        print("   Please train the model first (run Cell 10)")

NameError: name 'model' is not defined

In [1]:
# ==========================================
# QUICK INFERENCE ONLY (Skip training)
# ==========================================
import torch
import torch.nn as nn
from transformers import BertTokenizer, ViTModel, BertModel
import torchvision.transforms as transforms
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage
import os

# Config
class Config:
    CHECKPOINT_DIR = "/content/drive/MyDrive/vqa_checkpoints"
    VIT_MODEL = 'google/vit-base-patch16-224'
    BERT_MODEL = 'bert-base-uncased'
    IMG_SIZE = 224
    MAX_QUESTION_LEN = 64
    FUSION_SIZE = 512
    DROPOUT = 0.3
    USE_AMP = True

cfg = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model classes (same as above - simplified)
class SimpleViTBertVQA(nn.Module):
    def __init__(self, num_answers):
        super().__init__()
        self.vit = ViTModel.from_pretrained(cfg.VIT_MODEL)
        self.bert = BertModel.from_pretrained(cfg.BERT_MODEL)
        self.fusion = nn.Sequential(
            nn.Linear(768 + 768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
        )
        self.classifier = nn.Linear(256, num_answers)

    def forward(self, image, input_ids, mask):
        v = self.vit(image).last_hidden_state[:, 0, :]
        t = self.bert(input_ids, attention_mask=mask).last_hidden_state[:, 0, :]
        fused = self.fusion(torch.cat([v, t], dim=1))
        return self.classifier(fused)

# Load checkpoint
checkpoint_path = os.path.join(cfg.CHECKPOINT_DIR, 'best_model.pt')
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

model = SimpleViTBertVQA(len(checkpoint['idx_to_answer'])).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

tokenizer = BertTokenizer.from_pretrained(cfg.BERT_MODEL)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(f"✅ Ready! Best validation accuracy: {checkpoint['val_acc']:.2%}")

# Simple prediction function
def predict(image_path, question):
    image = transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
    tokens = tokenizer(question, max_length=64, padding='max_length', truncation=True, return_tensors='pt')
    with torch.no_grad():
        logits = model(image, tokens['input_ids'].to(device), tokens['attention_mask'].to(device))
        probs = torch.softmax(logits, dim=1)
    top_probs, top_idx = torch.topk(probs, 5)
    return [(checkpoint['idx_to_answer'][idx.item()], prob.item()) for idx, prob in zip(top_idx[0], top_probs[0])]

# Widgets
uploader = widgets.FileUpload(accept='image/*', multiple=False)
question_box = widgets.Text(placeholder='Ask a question...', description='Q:')
btn = widgets.Button(description='Predict', button_style='success')
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        if not uploader.value or not question_box.value:
            print("Upload image and enter question")
            return
        data = list(uploader.value.values())[0]
        with open('/tmp/img.jpg', 'wb') as f:
            f.write(data['content'])
        display(IPImage('/tmp/img.jpg', width=300))
        results = predict('/tmp/img.jpg', question_box.value)
        print(f"\n❓ {question_box.value}")
        print("\n🏆 Top predictions:")
        for i, (ans, conf) in enumerate(results):
            print(f"   {i+1}. {ans} ({conf:.1%})")

btn.on_click(on_click)
display(widgets.VBox([uploader, question_box, btn, out]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RuntimeError: Error(s) in loading state_dict for SimpleViTBertVQA:
	Missing key(s) in state_dict: "vit.embeddings.cls_token", "vit.embeddings.position_embeddings", "vit.embeddings.patch_embeddings.projection.weight", "vit.embeddings.patch_embeddings.projection.bias", "vit.encoder.layer.0.attention.attention.query.weight", "vit.encoder.layer.0.attention.attention.query.bias", "vit.encoder.layer.0.attention.attention.key.weight", "vit.encoder.layer.0.attention.attention.key.bias", "vit.encoder.layer.0.attention.attention.value.weight", "vit.encoder.layer.0.attention.attention.value.bias", "vit.encoder.layer.0.attention.output.dense.weight", "vit.encoder.layer.0.attention.output.dense.bias", "vit.encoder.layer.0.intermediate.dense.weight", "vit.encoder.layer.0.intermediate.dense.bias", "vit.encoder.layer.0.output.dense.weight", "vit.encoder.layer.0.output.dense.bias", "vit.encoder.layer.0.layernorm_before.weight", "vit.encoder.layer.0.layernorm_before.bias", "vit.encoder.layer.0.layernorm

In [42]:
# ==========================================
# CLEAR GPU MEMORY
# ==========================================
import torch
import gc

# Clear CUDA cache
torch.cuda.empty_cache()

# Force garbage collection
gc.collect()

# Clear all variables
if 'model' in globals():
    del model
if 'train_loader' in globals():
    del train_loader
if 'val_loader' in globals():
    del val_loader
if 'test_loader' in globals():
    del test_loader

# Reset CUDA memory stats
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

# Check memory
print(f"✅ Memory cleared!")
print(f"   Free memory: {torch.cuda.memory_reserved(device) / 1e9:.2f} GB reserved")
print(f"   Free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB / {torch.cuda.mem_get_info()[1] / 1e9:.2f} GB")

✅ Memory cleared!
   Free memory: 15.47 GB reserved
   Free: 0.01 GB / 15.64 GB
